> **Requires `MISTRAL_API_KEY`:** this notebook calls the Mistral API. Standard Colab will not run the code cells until you set the key (e.g. Colab **Secrets** → `MISTRAL_API_KEY`, then load it into `os.environ`, or `export` in your shell). Cells are not pre-executed in the repo copy.


# Paper-native RAG over a real arXiv PDF (RAGNav)

This notebook is the simplest end-user flow:
- download a PDF from a URL
- ingest in **paper mode** (pages + headings + cross-refs)
- route to pages, retrieve evidence
- answer (optionally with inline citations)

> Requirements: `pip install "ragnav[mistral,pdf]"` and `MISTRAL_API_KEY` set in your environment.


## Step 0: Install + set keys

In a notebook cell, run:

```bash
%pip install -q "ragnav[mistral,pdf]"
```

Set your key (recommended via env outside the notebook), or inside Python:

```python
import os
os.environ["MISTRAL_API_KEY"] = "..."
```


In [1]:
from ragnav.env import load_env
from ragnav.llm.mistral import MistralClient
from ragnav.papers import PaperRAG, PaperRAGConfig

# Loads .env if present
load_env()

llm = MistralClient()
cfg = PaperRAGConfig(
    max_pages=25,
    top_pages=4,
    follow_refs=True,
    include_next=True,
    k_final=10,
)


In [2]:
pdf_url = "https://arxiv.org/pdf/2507.13334.pdf"

paper = PaperRAG.from_pdf_url(
    pdf_url,
    llm=llm,
    pdf_name="2507.13334.pdf",
    cfg=cfg,
    cache_path="data/2507.13334.pdf",
)

query = "What is the paper's main contribution?"
print(paper.answer(query, cfg=cfg))


/Users/irfanalidv/.pyenv/versions/3.12.1/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


/Users/irfanalidv/.pyenv/versions/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

The paper's main contribution is the **formalization of Context Engineering as a discipline**, which re-conceptualizes the context for language models (LLMs) as a **dynamic, structured, and multifaceted information stream** rather than a static text prompt. It introduces a framework where context is assembled from modular components (e.g., retrieved knowledge, tools, memory, and system states) through optimized functions to maximize LLM performance. The paper also defines the **optimization problem of Context Engineering**, aiming to find the ideal set of context-generating functions to improve output quality across diverse tasks.


In [3]:
query = "What does Figure 1 show?"
try:
    print(paper.answer_cited(query, cfg=cfg))
except ValueError:
    # LLM formatting can fail strict per-sentence citation checks; fall back to plain answer.
    print(paper.answer(query, cfg=cfg))


Based on the provided context, the document does **not** contain a description or reference to **Figure 1**. The only figure mentioned is **Figure 2**, which shows a "Context Engineering Evolution Timeline."

Thus, the answer cannot be determined from the given context.
